# 02 - Retrieve and answer from a frozen corpus

Notebook này chỉ chạy **Retrieve & Answer**. Nó không có parser/OCR/VLM và không thể ingest thêm tài liệu.
Embedding mặc định được đọc từ manifest của corpus để bảo đảm document/query dùng cùng model.

Truy xuất: dense + BM25 → RRF (thực thể trong câu hỏi khớp hồ sơ tài liệu sẽ được boost mềm) → reranker →
mở rộng chunk lân cận cùng mục; bảng lớn được nạp đủ hàng và có file `.xlsx` để tải trong Chat UI.


## 1. Cài đặt và import source

In [ ]:
from datetime import datetime as _CellDateTime
import time as _cell_time

_cell_label = '02_retrieve_answer / 1. Cài đặt và import source'
_cell_started_at = _CellDateTime.now().astimezone()
_cell_started_perf = _cell_time.perf_counter()
print(f"[CELL START] {_cell_label} | {_cell_started_at.isoformat(timespec='seconds')}")
try:
    pass
    import os
    import subprocess
    import sys
    from pathlib import Path

    REPO_URL = "https://github.com/NhoDung/RAG-basic.git"
    REPO_DIR = Path("/kaggle/working/RAG-basic")

    if not (Path.cwd() / "rag_kaggle").exists():
        if REPO_DIR.exists():
            subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
        else:
            subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
        os.chdir(REPO_DIR)

    PROJECT_DIR = Path.cwd()
    sys.path.insert(0, str(PROJECT_DIR))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"])
    print("Project:", PROJECT_DIR)
finally:
    _cell_finished_at = _CellDateTime.now().astimezone()
    _cell_elapsed_seconds = _cell_time.perf_counter() - _cell_started_perf
    print(f"[CELL END] {_cell_label} | {_cell_finished_at.isoformat(timespec='seconds')}")
    print(f"[CELL ELAPSED_SECONDS] {_cell_label} | {_cell_elapsed_seconds:.3f}")


## 2. Chọn model trực tiếp

In [ ]:
from datetime import datetime as _CellDateTime
import time as _cell_time

_cell_label = '02_retrieve_answer / 2. Chọn model'
_cell_started_at = _CellDateTime.now().astimezone()
_cell_started_perf = _cell_time.perf_counter()
print(f"[CELL START] {_cell_label} | {_cell_started_at.isoformat(timespec='seconds')}")
try:
    pass
    import shutil

    # None = bắt buộc kế thừa embedding model/revision/query instruction từ corpus manifest.
    EMBEDDING_MODEL = None
    RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"  # None để tắt reranker
    GENERATION_MODEL = "Qwen/Qwen2.5-7B-Instruct"
    GENERATION_REVISION = None

    # Mở rộng context sau khi tìm thấy chunk.
    NEIGHBOR_CHUNKS = 1  # số chunk text kề nhau cùng mục được thêm vào hai phía
    MAX_TABLE_CHARS_IN_CONTEXT = 12000  # bảng lớn (chỉ index preview) được nạp lại tới ngưỡng này
    ENTITY_BOOST = True  # boost tài liệu có nhắc thực thể xuất hiện trong câu hỏi

    WORK_DIR = Path("/kaggle/working/rag_corpus")
    SESSION_DIR = Path("/kaggle/working/rag_session")

    # Thư mục corpus trong Kaggle Dataset được nén thành bundle để restore.
    CORPUS_SRC = "/kaggle/input/datasets/nhodng/data-test"
    CORPUS_BUNDLE = shutil.make_archive("/kaggle/working/corpus_bundle", "zip", root_dir=CORPUS_SRC)
    print("Bundle:", CORPUS_BUNDLE)
finally:
    _cell_finished_at = _CellDateTime.now().astimezone()
    _cell_elapsed_seconds = _cell_time.perf_counter() - _cell_started_perf
    print(f"[CELL END] {_cell_label} | {_cell_finished_at.isoformat(timespec='seconds')}")
    print(f"[CELL ELAPSED_SECONDS] {_cell_label} | {_cell_elapsed_seconds:.3f}")


## 3. Tạo retrieval runtime và khôi phục corpus

In [ ]:
from datetime import datetime as _CellDateTime
import time as _cell_time

_cell_label = '02_retrieve_answer / 3. Tạo retrieval runtime và khôi phục corpus'
_cell_started_at = _CellDateTime.now().astimezone()
_cell_started_perf = _cell_time.perf_counter()
print(f"[CELL START] {_cell_label} | {_cell_started_at.isoformat(timespec='seconds')}")
try:
    pass
    from rag_kaggle import (
        PipelineConfig, RetrievalAnswerPipeline, configure_retrieval_devices,
        inspect_resources, suggest_model_upgrades,
    )

    config = PipelineConfig.from_yaml(PROJECT_DIR / "configs" / "baseline.yaml")
    config.work_dir = WORK_DIR
    config.runtime.session_dir = str(SESSION_DIR)
    config.retrieval.dense_model = EMBEDDING_MODEL
    config.retrieval.reranker_enabled = RERANKER_MODEL is not None
    if RERANKER_MODEL:
        config.retrieval.reranker_model = RERANKER_MODEL
    config.generation.model = GENERATION_MODEL
    config.generation.revision = GENERATION_REVISION
    config.retrieval.neighbor_chunks = NEIGHBOR_CHUNKS
    config.retrieval.max_table_chars_in_context = MAX_TABLE_CHARS_IN_CONTEXT
    config.retrieval.entity_boost = ENTITY_BOOST

    resources = inspect_resources()
    print("Resources:", resources)
    print("Allocation:", configure_retrieval_devices(config))
    print("Suggestions:", suggest_model_upgrades("retrieval", resources, config))
    pipeline = RetrievalAnswerPipeline(config)

    if CORPUS_BUNDLE:
        restored = pipeline.restore_corpus_bundle(CORPUS_BUNDLE)
        print("Corpus:", restored["manifest"])
finally:
    _cell_finished_at = _CellDateTime.now().astimezone()
    _cell_elapsed_seconds = _cell_time.perf_counter() - _cell_started_perf
    print(f"[CELL END] {_cell_label} | {_cell_finished_at.isoformat(timespec='seconds')}")
    print(f"[CELL ELAPSED_SECONDS] {_cell_label} | {_cell_elapsed_seconds:.3f}")


## 4. Mở Chat UI

In [ ]:
from datetime import datetime as _CellDateTime
import time as _cell_time

_cell_label = '02_retrieve_answer / 4. Mở Chat UI'
_cell_started_at = _CellDateTime.now().astimezone()
_cell_started_perf = _cell_time.perf_counter()
print(f"[CELL START] {_cell_label} | {_cell_started_at.isoformat(timespec='seconds')}")
try:
    pass
    from rag_kaggle.ui import launch_chat_demo

    # Tab System cho phép upload corpus_bundle.zip từ máy cá nhân nếu cần.
    launch_chat_demo(pipeline, share=True, debug=False)
finally:
    _cell_finished_at = _CellDateTime.now().astimezone()
    _cell_elapsed_seconds = _cell_time.perf_counter() - _cell_started_perf
    print(f"[CELL END] {_cell_label} | {_cell_finished_at.isoformat(timespec='seconds')}")
    print(f"[CELL ELAPSED_SECONDS] {_cell_label} | {_cell_elapsed_seconds:.3f}")


## 5. API Python trực tiếp

In [ ]:
from datetime import datetime as _CellDateTime
import time as _cell_time

_cell_label = '02_retrieve_answer / 5. API Python trực tiếp'
_cell_started_at = _CellDateTime.now().astimezone()
_cell_started_perf = _cell_time.perf_counter()
print(f"[CELL START] {_cell_label} | {_cell_started_at.isoformat(timespec='seconds')}")
try:
    pass
    # result = pipeline.ask("Phí thường niên của thẻ Visa Gold là bao nhiêu?")
    # print(result["answer"], result["citations"])
    # print(result["query_entities"])  # thực thể trong câu hỏi khớp với hồ sơ tài liệu
    # print([c["table_file"] for c in result["contexts"] if c.get("table_file")])  # bảng đầy đủ (.xlsx)
    # metrics = pipeline.evaluate("/kaggle/input/my-evaluation/dataset.jsonl")
finally:
    _cell_finished_at = _CellDateTime.now().astimezone()
    _cell_elapsed_seconds = _cell_time.perf_counter() - _cell_started_perf
    print(f"[CELL END] {_cell_label} | {_cell_finished_at.isoformat(timespec='seconds')}")
    print(f"[CELL ELAPSED_SECONDS] {_cell_label} | {_cell_elapsed_seconds:.3f}")
